# Validation 평가 baseline

prediction과 300개 validation ground truth를 비교하여 전체 날짜 및 year/month/day 정확도를 측정하고 실패 사례를 추출한다. 이후 OCR/parser 실험을 동일한 기준으로 비교하기 위한 평가 코드다.

OCR 실행, 날짜 후보 탐색, 날짜 순서·국가·언어 추론 및 prediction 보정은 하지 않는다. 실제 prediction이 없으면 성능 수치를 생성하지 않는다.

## 1. 라이브러리 및 경로
`PREDICTION_PATH` 한 곳에서 평가할 파일을 변경한다. CSV는 문자열로 읽고 NONE과 image_id의 선행 0을 보존한다. 입력 파일은 읽기만 한다.

In [1]:
from pathlib import Path
import hashlib
import re
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((root for root in (cwd, *cwd.parents)
                     if (root / "labels" / "labels_300.csv").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("labels/labels_300.csv가 있는 프로젝트에서 실행하세요.")
GROUND_TRUTH_PATH = PROJECT_ROOT / "labels" / "labels_300.csv"
PREDICTION_PATH = PROJECT_ROOT / "outputs" / "validation_predictions.csv"
REQUIRED = ["image_id", "year", "month", "day", "final_date"]
FIELDS = ["year", "month", "day"]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def read_labels(path):
    return pd.read_csv(path, dtype=str, encoding="utf-8-sig",
                       keep_default_na=False, na_filter=False)

original_gt_sha256 = sha256(GROUND_TRUTH_PATH)
ground_truth = read_labels(GROUND_TRUTH_PATH)
original_ground_truth = ground_truth.copy(deep=True)
prediction = None
original_prediction = None
original_prediction_sha256 = None
if PREDICTION_PATH.is_file():
    original_prediction_sha256 = sha256(PREDICTION_PATH)
    prediction = read_labels(PREDICTION_PATH)
    original_prediction = prediction.copy(deep=True)
else:
    print("prediction 파일이 아직 없습니다. 팀원 결과 생성 후 평가를 실행하세요.")
    print("예정 경로:", PREDICTION_PATH)


prediction 파일이 아직 없습니다. 팀원 결과 생성 후 평가를 실행하세요.
예정 경로: C:\Users\jsw58\Desktop\itda-ocr\outputs\validation_predictions.csv


## 2. 비교 기준 및 입력 검증
비교용 복사본에서 앞뒤 공백을 제거하고 `None/none/NONE` 문자열을 `NONE`으로 통일한다. 날짜 요소가 ASCII 숫자(`0–9`)로만 구성되면 정수 의미로 비교한다. 따라서 `04`와 `4`는 같다. 소수, 부호, 기타 문자열은 숫자로 해석하거나 보정하지 않는다. 빈 문자열은 NONE과 다르다.

image_id는 앞뒤 공백만 제거하며 선행 0과 대소문자를 보존한다. 필수 컬럼 누락, 빈 ID, 중복 ID는 평가를 중단한다(임의 행 선택 없음). GT에 없는 ID는 보고 후 평가에서 제외한다. 누락 prediction은 overall 평가에서 오답이다. 예시는 최대 10개만 출력한다.

In [2]:
def normalize_component(value):
    value = value.strip()
    if value.casefold() == "none":
        return "NONE"
    if re.fullmatch(r"[0-9]+", value):
        return value.lstrip("0") or "0"
    return value

# 정규화 규약만 확인하며 예측 데이터나 성능 수치는 만들지 않는다.
assert normalize_component(" 04 ") == normalize_component("4")
assert normalize_component("None") == normalize_component("none") == "NONE"
assert normalize_component("") != "NONE"
assert normalize_component("4.0") == "4.0"

def validate_structure(frame, name):
    missing_columns = [c for c in REQUIRED if c not in frame.columns]
    print(f"{name} 필수 컬럼 누락: {len(missing_columns)}개; {missing_columns}")
    if "image_id" not in frame.columns:
        print(f"{name}: image_id 컬럼이 없어 ID 검증 불가")
        return False
    ids = frame["image_id"].str.strip()
    blank = ids.eq("")
    duplicate = ids.ne("") & ids.duplicated(keep=False)
    duplicate_ids = ids[duplicate].drop_duplicates()
    print(f"{name} 빈 image_id: {int(blank.sum())}행; 행 인덱스 예시: {frame.index[blank].tolist()[:10]}")
    print(f"{name} 중복 image_id: {len(duplicate_ids)}개 / 관련 {int(duplicate.sum())}행; 예시: {duplicate_ids.tolist()[:10]}")
    return not missing_columns and not blank.any() and not duplicate.any()

def comparison_copy(frame):
    result = frame.loc[:, REQUIRED].copy(deep=True)
    result["image_id"] = result["image_id"].str.strip()
    for field in FIELDS:
        result[field] = result[field].map(normalize_component)
    return result

gt_valid = validate_structure(ground_truth, "Ground truth")
pred_valid = False
gt_eval = None
pred_eval = None
partial_none_mask = None
if gt_valid:
    gt_eval = comparison_copy(ground_truth)
    none_count = gt_eval[FIELDS].eq("NONE").sum(axis=1)
    partial_none_mask = none_count.ge(1)
    print(f"Ground truth: {len(gt_eval)}행 / 고유 image_id {gt_eval['image_id'].nunique()}개")
    print(f"부분 NONE (1~2개): {int(none_count.between(1, 2).sum())}개")
    print(f"NONE 포함 (1개 이상, 별도 평가 대상): {int(partial_none_mask.sum())}개")
    print(f"전체 NONE (3개): {int(none_count.eq(3).sum())}개")
if prediction is not None:
    pred_valid = validate_structure(prediction, "Prediction")
    if "image_id" in ground_truth.columns and "image_id" in prediction.columns:
        gt_ids = set(ground_truth["image_id"].str.strip()) - {""}
        pred_ids = set(prediction["image_id"].str.strip()) - {""}
        unknown_ids = sorted(pred_ids - gt_ids)
        missing_ids = sorted(gt_ids - pred_ids)
        print(f"GT에 없는 prediction image_id: {len(unknown_ids)}개; 예시: {unknown_ids[:10]}")
        print(f"prediction에 빠진 GT image_id: {len(missing_ids)}개; 예시: {missing_ids[:10]}")
    if pred_valid:
        pred_eval = comparison_copy(prediction)

evaluation_ready = gt_valid and pred_valid
if not gt_valid or (prediction is not None and not pred_valid):
    print("입력 형식 오류로 정확도를 계산하지 않습니다. 위 문제를 해결한 뒤 전체 셀을 다시 실행하세요.")


Ground truth 필수 컬럼 누락: 0개; []
Ground truth 빈 image_id: 0행; 행 인덱스 예시: []
Ground truth 중복 image_id: 0개 / 관련 0행; 예시: []
Ground truth: 300행 / 고유 image_id 300개
부분 NONE (1~2개): 18개
NONE 포함 (1개 이상, 별도 평가 대상): 18개
전체 NONE (3개): 0개


## 3. Coverage 및 정확도
GT 기준 left join을 사용하고 ID가 일대일 대응하는지 검증한다. **Overall accuracy를 우선 확인한다**: 전체 GT가 분모이며 누락 prediction은 오답이다. Matched accuracy는 GT와 매칭된 prediction만 분모로 한다. 분모가 0이면 N/A로 표시한다.

Final date accuracy는 제공된 `final_date` 문자열을 파싱하지 않고 **정규화된 year/month/day 세 요소의 동시 일치**로 계산한다. `final_date` 원문은 실패 사례 참고용으로 유지한다. 이 규칙은 부분 NONE에도 동일하게 적용한다.

In [3]:
evaluation = None
accuracy_table = None
coverage = None
partial_none_result = None

def ratio(correct, total):
    return correct / total if total else None

def percent(value):
    return "N/A (분모 0)" if value is None else f"{value:.2%}"

if evaluation_ready:
    evaluation = gt_eval.merge(pred_eval, on="image_id", how="left",
                               suffixes=("_true", "_pred"), indicator=True,
                               validate="one_to_one")
    matched = evaluation["_merge"].eq("both")
    evaluation["missing_prediction"] = ~matched
    for field in FIELDS:
        evaluation[f"{field}_correct"] = (
            matched & evaluation[f"{field}_true"].eq(evaluation[f"{field}_pred"]))
    evaluation["final_date_correct"] = evaluation[
        [f"{field}_correct" for field in FIELDS]].all(axis=1)
    total = len(gt_eval)
    prediction_images = pred_eval["image_id"].nunique()
    matched_count = int(matched.sum())
    missing_count = total - matched_count
    coverage = ratio(matched_count, total)
    rows = []
    for field in [*FIELDS, "final_date"]:
        correct = int(evaluation[f"{field}_correct"].sum())
        rows.append({"metric": field, "correct": correct,
                     "overall_denominator": total,
                     "overall_accuracy": ratio(correct, total),
                     "matched_denominator": matched_count,
                     "matched_accuracy": ratio(correct, matched_count)})
    accuracy_table = pd.DataFrame(rows).set_index("metric")
    print(f"Ground truth: {total}; Prediction 고유 ID: {prediction_images}; Matched: {matched_count}")
    print(f"누락 prediction: {missing_count}; Coverage: {percent(coverage)}")
    print("GT에 없는 prediction ID는 정확도와 coverage의 분자에 포함하지 않습니다.")
    display(accuracy_table)


## 4. 실패 사례
누락 prediction을 포함한 전체 날짜 오답을 `failure_cases`에 보관한다. 아래 표는 최대 20건만 보여주며 긴 문자열은 줄바꿈한다. `true_*` / `pred_*`는 입력 원문이고 error 판정은 정규화된 비교 결과다. 누락 예측의 각 요소는 오답으로 표시한다.

In [4]:
failure_cases = None
if evaluation_ready:
    true_raw = ground_truth.loc[:, REQUIRED].copy()
    true_raw["image_id"] = true_raw["image_id"].str.strip()
    true_raw["notes"] = ground_truth["notes"] if "notes" in ground_truth.columns else ""
    true_raw = true_raw.rename(columns={c: f"true_{c}" for c in REQUIRED if c != "image_id"})
    pred_raw = prediction.loc[:, REQUIRED].copy()
    pred_raw["image_id"] = pred_raw["image_id"].str.strip()
    pred_raw = pred_raw.rename(columns={c: f"pred_{c}" for c in REQUIRED if c != "image_id"})
    details = true_raw.merge(pred_raw, on="image_id", how="left", validate="one_to_one")
    errors = evaluation[["image_id", "missing_prediction", "final_date_correct"]].copy()
    for field in FIELDS:
        errors[f"{field}_error"] = ~evaluation[f"{field}_correct"]
    details = details.merge(errors, on="image_id", validate="one_to_one")
    details["failure_type"] = details.apply(
        lambda row: "missing_prediction" if row["missing_prediction"] else
        " | ".join(f"{field}_error" for field in FIELDS if row[f"{field}_error"]), axis=1)
    failure_columns = ["image_id", "true_year", "pred_year", "true_month", "pred_month",
                       "true_day", "pred_day", "true_final_date", "pred_final_date", "notes",
                       "year_error", "month_error", "day_error", "missing_prediction", "failure_type"]
    failure_cases = details.loc[~details["final_date_correct"], failure_columns].copy()
    print(f"Failure cases: {len(failure_cases)}건 (미리보기 최대 20건, 전체 변수: failure_cases)")
    preview_columns = ["image_id", "true_final_date", "pred_final_date", "notes", "failure_type"]
    with pd.option_context("display.max_rows", 20, "display.max_columns", 15,
                           "display.max_colwidth", 120):
        display(failure_cases.loc[:, preview_columns].head(20).style.hide(axis="index")
                .format(na_rep="(prediction 없음)", escape="html")
                .set_table_attributes('style="width:100%; table-layout:fixed;"')
                .set_properties(**{"white-space": "normal", "overflow-wrap": "anywhere",
                                   "vertical-align": "top", "text-align": "left"}))


## 5. 부분 NONE 별도 평가
GT의 year/month/day 중 하나 이상이 NONE인 사례를 평가한다. 현재 부분 NONE 18건을 포함하며, 전체 NONE 사례가 추가되면 함께 포함한다. 이 집단 전체가 분모이고 누락 prediction도 오답이다. 별도로 matched 분모도 제공한다.

In [5]:
if evaluation_ready:
    subset = evaluation.loc[evaluation[[f"{f}_true" for f in FIELDS]].eq("NONE").any(axis=1)]
    subset_count = len(subset)
    subset_correct = int(subset["final_date_correct"].sum())
    subset_matched = int((~subset["missing_prediction"]).sum())
    partial_none_result = {"cases": subset_count, "correct": subset_correct,
                           "accuracy": ratio(subset_correct, subset_count),
                           "matched_cases": subset_matched,
                           "matched_accuracy": ratio(subset_correct, subset_matched)}
    print(f"Partial-NONE: {subset_correct}/{subset_count}; accuracy: {percent(partial_none_result['accuracy'])}")
    print(f"Matched Partial-NONE: {subset_correct}/{subset_matched}; accuracy: {percent(partial_none_result['matched_accuracy'])}")


## 6. 원본 보호 검증 및 최종 요약
입력 CSV의 SHA-256과 로드 직후 DataFrame 복사본을 비교한다. 실제 prediction이 없거나 입력 형식 검증에 실패하면 정확도를 출력하지 않는다.

In [6]:
assert sha256(GROUND_TRUTH_PATH) == original_gt_sha256, "원본 GT CSV 변경 감지"
assert ground_truth.equals(original_ground_truth), "원본 GT DataFrame 변경 감지"
if prediction is not None:
    assert sha256(PREDICTION_PATH) == original_prediction_sha256, "원본 prediction CSV 변경 감지"
    assert prediction.equals(original_prediction), "원본 prediction DataFrame 변경 감지"
print("원본 CSV SHA-256 및 DataFrame 변경 없음")

if evaluation_ready:
    print(f"Ground truth images: {total}")
    print(f"Prediction images: {prediction_images} (GT 외 ID 포함)")
    print(f"Matched images: {matched_count}")
    print(f"Missing predictions: {missing_count}")
    print(f"Coverage: {percent(coverage)} ({matched_count}/{total})")
    print(f"Overall accuracy 분모: {total} (누락 prediction은 오답; 성능 확인 시 우선)")
    for field in [*FIELDS, "final_date"]:
        correct = int(evaluation[f"{field}_correct"].sum())
        print(f"Overall {field.replace('_', ' ')} accuracy: {percent(ratio(correct, total))} ({correct}/{total})")
    print(f"Matched accuracy 분모: {matched_count}")
    for field in [*FIELDS, "final_date"]:
        correct = int(evaluation[f"{field}_correct"].sum())
        print(f"{field.replace('_', ' ').capitalize()} accuracy: {percent(ratio(correct, matched_count))} ({correct}/{matched_count})")
    print(f"Partial-NONE accuracy: {percent(partial_none_result['accuracy'])} "
          f"({partial_none_result['correct']}/{partial_none_result['cases']}; 누락 포함)")
    print(f"Failure cases: {len(failure_cases)}")
elif prediction is None:
    print("prediction 파일이 아직 없습니다. 팀원 결과 생성 후 평가를 실행하세요.")
else:
    print("입력 형식 검증 실패: 정확도 계산을 생략했습니다.")


원본 CSV SHA-256 및 DataFrame 변경 없음
prediction 파일이 아직 없습니다. 팀원 결과 생성 후 평가를 실행하세요.
